# Sentence-BERT content recommender — pipeline validation

**Goal of this notebook** (examiner-facing demo)

1. Build a content-aware Stage 1 recommender using Sentence-BERT.
2. Show it produces **non-zero cold-track recall** where pure-CF models (popularity, MF, EASE, BPR) score exactly 0 by construction.
3. Validate the dual-track evaluation protocol described in `docs/data_decisions.md` §1b.

**Why this matters for PantryPlate**

The cold-track test set (from Majumder et al.'s pre-split) holds out *recipes with zero raters in train* — pure new items. A content-aware model is the only thing that can reach them. Demonstrating any non-zero number on cold is the architectural win that justifies the hybrid Stage 1 in our final proposal.

**Roadmap**

1. Load training data
2. Inspect what we embed (recipe text)
3. Fit Sentence-BERT (encode + cache)
4. Sanity check — what does a user's top-5 look like?
5. Warm-track evaluation (vs popularity baseline)
6. Cold-track evaluation (vs popularity baseline) — the real test
7. Summary

## Setup

In [1]:
import os
import sys
from pathlib import Path

# Make the notebook robust to whichever directory it's launched from.
def _ensure_project_root():
    cwd = Path(os.getcwd()).resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            os.chdir(candidate)
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError("Could not locate the PantryPlate project root.")

PROJECT_ROOT = _ensure_project_root()
print(f"Project root: {PROJECT_ROOT}")

import time
from contextlib import contextmanager

import numpy as np
import pandas as pd

from src.data.loader import load_train_interactions, load_recipes, time_based_split
from src.models.sentence_bert import SentenceBERTRecommender, _build_recipe_text
from src.models.popularity import PopularityRecommender
from src.eval.harness import evaluate

@contextmanager
def timer(label):
    t0 = time.time()
    yield
    print(f"  [{time.time() - t0:.1f}s] {label}")

Project root: /Users/ikhyvicky/Documents/MITB_stuff/CS608Project2


## 1. Load training data

We use the authors' published `interactions_train.csv` (locked decision #1). Note that this file is **not** pre-filtered to ≥5 ratings — it has a wide activity spread.

In [2]:
with timer("loaded train interactions"):
    full_train = load_train_interactions()

# CRITICAL: for warm-track eval, fit on the post-LOO train, not the full train.
# Otherwise exclude_seen=True will exclude the held-out test item from recommendations
# (it's "seen" in the full train) and Recall@K is silently 0.
# See docs/eval_harness_usage.md "Common mistakes" section.
with timer("time-based LOO split"):
    train, warm_holdout = time_based_split(full_train, holdout_per_user=1)

print(f"Full train rows:    {len(full_train):>10,}")
print(f"  ↳ train (post-split, what we fit on): {len(train):>10,} rows")
print(f"  ↳ warm holdout (what the harness scores against): {len(warm_holdout):>10,} rows")
print(f"Unique users in train:   {train['user_id'].nunique():>10,}")
print(f"Unique recipes in train: {train['recipe_id'].nunique():>10,}")
print(f"\nRating distribution in train:")
print(train['rating'].value_counts().sort_index())
print(f"\nPositives (rating >= 4): {(train['rating'] >= 4).sum():,}  ({(train['rating'] >= 4).mean()*100:.1f}%)")

  [0.3s] loaded train interactions
  [0.2s] time-based LOO split
Full train rows:       681,944
  ↳ train (post-split, what we fit on):    657,562 rows
  ↳ warm holdout (what the harness scores against):     24,382 rows
Unique users in train:       24,961


Unique recipes in train:    158,340

Rating distribution in train:
rating
1.0      3341
2.0      6852
3.0     25781
4.0    123791
5.0    497797
Name: count, dtype: int64

Positives (rating >= 4): 621,588  (94.5%)


## 2. Inspect what we'll embed (recipe text)

Each recipe becomes a single string: `name | ingredients | tags`. This is what Sentence-BERT actually sees.

In [3]:
recipes = load_recipes()
sample_recipes = recipes.sample(3, random_state=42)

for _, row in sample_recipes.iterrows():
    text = _build_recipe_text(row)
    print(f"recipe_id={row['id']}")
    print(f"  text: {text[:200]}{'...' if len(text) > 200 else ''}")
    print()

recipe_id=94947
  text: crab filled crescent snacks | crabmeat, cream cheese, green onions, garlic salt, refrigerated crescent dinner rolls, egg yolk, water, sesame seeds, sweet and sour sauce | time-to-make, course, main-in...

recipe_id=429010
  text: curried bean salad | garbanzo beans, black beans, onion, ginger paste, mild curry powder, dried cilantro, lemon juice, diced tomatoes, creamed corn, cooked brown rice, rice cakes, raisins | curries, 3...

recipe_id=277542
  text: delicious steak with onion marinade | olive oil, red onion, light brown sugar, balsamic vinegar, steaks | lactose, 30-minutes-or-less, time-to-make, course, main-ingredient, preparation, main-dish, be...



## 3. Fit Sentence-BERT

This encodes all 231K recipes into 384-dim embeddings using `all-MiniLM-L6-v2` (lightweight ~80MB model) and builds user profile vectors from each user's positive ratings.

**First run**: ~5–10 min (encodes + caches to `data/processed/recipe_sbert_*.npy`).  
**Subsequent runs**: ~10s (loads cache).

In [4]:
with timer("Sentence-BERT fit"):
    sbert = SentenceBERTRecommender(batch_size=256)
    sbert.fit(train)  # NOTE: train_part from the LOO split, not full_train

print(f"\nRecipe matrix shape:  {sbert._recipe_matrix.shape}  (n_recipes, embedding_dim)")
print(f"User profiles built:  {len(sbert._user_vectors):,}")
print(f"Users in train:       {train['user_id'].nunique():,}")
print(f"  ↳ Users without profile (no positives in train_part): {train['user_id'].nunique() - len(sbert._user_vectors):,}")
print(f"  ↳ These users will fall back to popularity at recommend time.")

  [19.9s] Sentence-BERT fit

Recipe matrix shape:  (231637, 384)  (n_recipes, embedding_dim)
User profiles built:  24,225
Users in train:       24,961
  ↳ Users without profile (no positives in train_part): 736
  ↳ These users will fall back to popularity at recommend time.


In [5]:
with timer("Popularity baseline fit"):
    pop = PopularityRecommender().fit(train)  # same train_part

  [0.2s] Popularity baseline fit


## 4. Sanity check — what does a user's top-5 look like?

Pick a user with many positives, look at:
- A few recipes they actually rated 5
- Sentence-BERT's top-5 recommendations for them

We want the recommended recipes to feel thematically similar to what they liked — not identical (those are excluded as `exclude_seen=True`), but in the same flavor / ingredient neighborhood.

In [6]:
# Pick a user with lots of positives (in train_part)
user_positives = train[train['rating'] >= 4].groupby('user_id').size().sort_values(ascending=False)
demo_user = int(user_positives.index[100])  # 101st most-active user
print(f"Demo user: {demo_user}  ({user_positives[demo_user]} positives in train_part)")

# Their 5 most recent positives (in train_part)
user_train = train[(train['user_id'] == demo_user) & (train['rating'] >= 4)]
user_train = user_train.sort_values('date', ascending=False).head(5)
name_lookup = recipes.set_index(recipes['id'].astype(int))['name']

print("\nTheir 5 most recent 4+ star recipes (what we know they liked):")
for _, row in user_train.iterrows():
    print(f"  {row['recipe_id']:>10}  {name_lookup.get(int(row['recipe_id']), '(unknown)')}")

# Sentence-BERT's top-5 recommendations
recs = sbert.recommend(demo_user, k=5, exclude_seen=True)
print("\nSentence-BERT's top-5 recommendations (exclude_seen=True):")
for rid in recs:
    print(f"  {rid:>10}  {name_lookup.get(rid, '(unknown)')}")

Demo user: 52282  (765 positives in train_part)

Their 5 most recent 4+ star recipes (what we know they liked):
      203834  mom s pork tenderloin
       62469  pork with a blue cheese apple and mustard sauce
      116269  dutch slavinken   1
       34919  brats with whiskey glazed onions
      408682  mexican ground beef pie

Sentence-BERT's top-5 recommendations (exclude_seen=True):
      183920  peanutty spicy noodle salad
      116496  cheddar and veggie bread pudding
       24945  banana ketchup
      115822  sweet and sour seashells
       64887  curried carrot hummus crab cakes


## 5. Warm-track evaluation

**Track A** — the standard recsys benchmark. Hold out each user's most-recent positive from train; rank candidates; measure Recall@K / NDCG@K / MRR.

We sample 2,000 users for speed (same default the harness uses). Popularity baseline run alongside for comparison.

In [7]:
with timer("Sentence-BERT warm eval"):
    sbert_warm = evaluate(sbert, track="warm", n_users=2000, seed=42)

with timer("Popularity warm eval"):
    pop_warm = evaluate(pop, track="warm", n_users=2000, seed=42)

warm_df = pd.DataFrame({
    "Sentence-BERT": [sbert_warm[k] for k in ['recall@5', 'recall@10', 'recall@20', 'ndcg@10', 'mrr']],
    "Popularity":    [pop_warm[k]   for k in ['recall@5', 'recall@10', 'recall@20', 'ndcg@10', 'mrr']],
}, index=['Recall@5', 'Recall@10', 'Recall@20', 'NDCG@10', 'MRR']) * 100
warm_df['Δ (pp)'] = warm_df['Sentence-BERT'] - warm_df['Popularity']
warm_df.round(2)

  [16.4s] Sentence-BERT warm eval
  [0.1s] Popularity warm eval


,Sentence-BERT,Popularity,Δ (pp)
Recall@5,0.05,1.85,-1.80
Recall@10,0.15,2.95,-2.80
Recall@20,0.20,4.50,-4.30
NDCG@10,0.08,1.39,-1.30
MRR,0.07,1.02,-0.95


## 6. Cold-track evaluation (the real test)

**Track B** — the authors' published test set holds out items with **zero raters in train**. Pure-CF models score exactly 0 here by construction. A content-aware model should produce a non-zero number.

This is the architectural justification for hybridizing CF + content in Stage 1.

In [8]:
with timer("Sentence-BERT cold eval"):
    sbert_cold = evaluate(sbert, track="cold", seed=42)

with timer("Popularity cold eval"):
    pop_cold = evaluate(pop, track="cold", seed=42)

cold_df = pd.DataFrame({
    "Sentence-BERT": [sbert_cold[k] for k in ['recall@5', 'recall@10', 'recall@20', 'ndcg@10', 'mrr']],
    "Popularity":    [pop_cold[k]   for k in ['recall@5', 'recall@10', 'recall@20', 'ndcg@10', 'mrr']],
}, index=['Recall@5', 'Recall@10', 'Recall@20', 'NDCG@10', 'MRR']) * 100
cold_df['Δ (pp)'] = cold_df['Sentence-BERT'] - cold_df['Popularity']
cold_df.round(3)

  [79.9s] Sentence-BERT cold eval
  [0.2s] Popularity cold eval


,Sentence-BERT,Popularity,Δ (pp)
Recall@5,0.067,0.0,0.067
Recall@10,0.087,0.0,0.087
Recall@20,0.144,0.0,0.144
NDCG@10,0.056,0.0,0.056
MRR,0.050,0.0,0.050


## 7. Summary

In [9]:
print("=" * 60)
print("PIPELINE VALIDATION SUMMARY")
print("=" * 60)

warm_delta = (sbert_warm['recall@10'] - pop_warm['recall@10']) * 100
cold_delta = (sbert_cold['recall@10'] - pop_cold['recall@10']) * 100

print(f"\nTrack A — warm (sample of 2000 users):")
print(f"  Sentence-BERT Recall@10: {sbert_warm['recall@10']*100:6.2f}%")
print(f"  Popularity    Recall@10: {pop_warm['recall@10']*100:6.2f}%")
print(f"  Δ vs popularity:         {warm_delta:+.2f} pp")

print(f"\nTrack B — cold (all {sbert_cold['n_users_evaluated']:,} cold-item users):")
print(f"  Sentence-BERT Recall@10: {sbert_cold['recall@10']*100:6.2f}%")
print(f"  Popularity    Recall@10: {pop_cold['recall@10']*100:6.2f}%   <- 0% by construction (cold items have no rater history)")
print(f"  Δ vs popularity:         {cold_delta:+.2f} pp")

print()
print("Findings:")
if sbert_cold['recall@10'] > 0:
    print(f"  ✓ Content-aware model produces non-zero cold Recall@10 — the architectural")
    print(f"    payoff for hybridizing CF + content in Stage 1 is validated.")
else:
    print(f"  ✗ Cold Recall@10 = 0 — something is wrong; investigate before proceeding.")
if warm_delta > 0:
    print(f"  ✓ Sentence-BERT also beats popularity on warm Recall@10 (+{warm_delta:.2f} pp).")
else:
    print(f"  ⓘ Sentence-BERT is below popularity on warm ({warm_delta:+.2f} pp). Expected for a")
    print(f"    content-only model — its job is to win on cold. CF models (EASE, BPR) will")
    print(f"    handle the warm track.")

PIPELINE VALIDATION SUMMARY

Track A — warm (sample of 2000 users):
  Sentence-BERT Recall@10:   0.15%
  Popularity    Recall@10:   2.95%
  Δ vs popularity:         -2.80 pp

Track B — cold (all 10,393 cold-item users):
  Sentence-BERT Recall@10:   0.09%
  Popularity    Recall@10:   0.00%   <- 0% by construction (cold items have no rater history)
  Δ vs popularity:         +0.09 pp

Findings:
  ✓ Content-aware model produces non-zero cold Recall@10 — the architectural
    payoff for hybridizing CF + content in Stage 1 is validated.
  ⓘ Sentence-BERT is below popularity on warm (-2.80 pp). Expected for a
    content-only model — its job is to win on cold. CF models (EASE, BPR) will
    handle the warm track.


## What's next

1. **Stage 2 reranker scaffold** — wire Sentence-BERT into a Stage 2 candidate pipeline with the pantry / nutrition / diet score functions.
2. **Hybrid model** — combine Sentence-BERT (content) with a CF model (EASE or BPR) via `α·CF + (1-α)·content`. Tag SVD content model is also a candidate alternative content side.
3. **α-sweep experiments** — generate the headline plot for the proposal.

See `docs/week2_onboarding.md` §4b for the live coordination table.